# Data Analysis for Drivetrain and ICE Datasets

This notebook provides a modular skeleton for preprocessing and analyzing CSV files from the `Data_Drivetrain` and `Data_ICE` directories. It handles the specific CSV structure (comments, units rows, tab delimiters) and provides functions for calculating min/max values and visualizing the data.

## 1. Imports and Setup

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# Set pandas display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Directory Paths
DATA_DRIVETRAIN_DIR = '../internal_lstm_models/data_drivetrain_processed'
DATA_ICE_DIR = '../internal_lstm_models/data_ice_processed'


## 2. Helper Functions for Data Loading and Preprocessing

These functions handle finding files, loading CSVs (accounting for different parsing rules, like skipped lines or delimiters), and basic preprocessing.

In [2]:
def get_csv_files(directory):
    """Returns a list of all CSV files in the given directory."""
    return sorted(glob.glob(os.path.join(directory, '*.csv')))

def load_and_preprocess_csv(filepath):
    """
    Loads a single CSV file and performs basic preprocessing.
    Accounts for the specific layout where line 0 is a comment, 
    line 2 specifies names, and line 3 specifies units.
    """
    try:
        # The files use tabs (\t) as delimiters.
        # Line 0 is a comment.
        # Line 1 is empty.
        # Line 2 contains headers.
        # Line 3 contains units.
        df = pd.read_csv(filepath, sep=',')
        
        # Drop the first row which contains units instead of numbers
        # df = df.iloc[1:].reset_index(drop=True)  # no longer needed for processed data
        
        # Convert all columns to numeric, coercing errors (like strings) to NaN
        df = df.apply(pd.to_numeric, errors='coerce')
        
        # Optionally clean up column names (removing ;Units if present)
        # df.columns = [col.split(';')[0] for col in df.columns]
        
        return df
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

## 3. Aggregating Min/Max Values

The following functions allow you to extract the minimum and maximum values for each column, either for a single file or aggregated across all files in a directory.

In [3]:
def get_min_max_single_file(filepath):
    """Returns the min and max for each column in a single CSV file as a DataFrame."""
    df = load_and_preprocess_csv(filepath)
    if df is not None and not df.empty:
        return pd.DataFrame({"Min": df.min(), "Max": df.max()})
    return None


def get_min_max_all_files(directory):
    """
    Iterates over all CSV files in the directory and finds the absolute
    min and max for each column across all files.
    """
    all_files = get_csv_files(directory)
    print(f"Found {len(all_files)} files in {directory}")

    global_min = pd.Series(dtype=float)
    global_max = pd.Series(dtype=float)

    for i, file in enumerate(all_files):
        df = load_and_preprocess_csv(file)
        if df is not None and not df.empty:
            current_min = df.min()
            current_max = df.max()

            if global_min.empty:
                global_min = current_min
                global_max = current_max
            else:
                # Align indices and find absolute min/max across files
                global_min = np.minimum(global_min, current_min)
                global_max = np.maximum(global_max, current_max)

        if (i + 1) % 20 == 0:
            print(f"Processed {i+1}/{len(all_files)} files...")

    print(f"Finished processing all {len(all_files)} files.")
    return pd.DataFrame({"Global_Min": global_min, "Global_Max": global_max})

## 3b. Full Basic Statistics per Column

In [4]:
def get_basic_stats_single_file(filepath):
    """
    Returns basic descriptive statistics (count, mean, std, min, 25%, 50%, 75%, max)
    for each column in a single CSV file.
    """
    df = load_and_preprocess_csv(filepath)
    if df is not None and not df.empty:
        return df.describe().T  # Transpose so columns are rows for easy reading
    return None


def get_basic_stats_all_files(directory, sample_size=None):
    """
    Aggregates basic descriptive statistics across all CSV files in a directory.

    Concatenates all loaded DataFrames and computes statistics on the combined
    data for accuracy. If memory is a concern, pass `sample_size` to randomly
    sample that many rows per file instead of loading the entire file.

    Parameters
    ----------
    directory : str
        Path to the directory containing CSV files.
    sample_size : int or None
        If set, randomly sample this many rows per file before concatenating.

    Returns
    -------
    pd.DataFrame
        Transposed describe() output: one row per original column, statistics
        as columns (count, mean, std, min, 25%, 50%, 75%, max).
    """
    all_files = get_csv_files(directory)
    print(f"Found {len(all_files)} files in {directory}")

    frames = []
    for i, file in enumerate(all_files):
        df = load_and_preprocess_csv(file)
        if df is not None and not df.empty:
            if sample_size is not None:
                df = df.sample(n=min(sample_size, len(df)), random_state=42)
            frames.append(df)

        if (i + 1) % 20 == 0:
            print(f"Loaded {i + 1}/{len(all_files)} files...")

    if not frames:
        print("No data loaded.")
        return None

    combined = pd.concat(frames, ignore_index=True)
    print(f"Finished loading all {len(all_files)} files. Total rows: {len(combined):,}")
    return combined.describe().T


## 4. Run Analysis: Basic Testing

Let's test our functions on the `Data_Drivetrain` and `Data_ICE` directories.

In [ ]:
# Example: Get min/max for a single file (Drivetrain)
drivetrain_files = get_csv_files(DATA_DRIVETRAIN_DIR)
if drivetrain_files:
    sample_file = drivetrain_files[0]
    print(f"Analyzing single file: {os.path.basename(sample_file)}")
    # min_max_df = get_min_max_single_file(sample_file)
    # display(min_max_df)

    basic_stats_df = get_basic_stats_single_file(sample_file)
    display(basic_stats_df)

In [ ]:
# Example: Get global min/max for ALL Drivetrain files
# (This might take a minute depending on the number of files)
print("Analyzing ALL Drivetrain files...")
# global_drivetrain_min_max = get_min_max_all_files(DATA_DRIVETRAIN_DIR)
# display(global_drivetrain_min_max.head(10))

global_drivetrain_basic_stats = get_basic_stats_all_files(DATA_DRIVETRAIN_DIR)
display(global_drivetrain_basic_stats)

In [5]:
# get basic stats for all ice files
print("\nAnalyzing ALL ICE files...")
global_ice_basic_stats = get_basic_stats_all_files(DATA_ICE_DIR)
display(global_ice_basic_stats)


Analyzing ALL ICE files...
Found 384 files in ../internal_lstm_models/data_ice_processed
Loaded 20/384 files...
Loaded 40/384 files...
Loaded 60/384 files...
Loaded 80/384 files...
Loaded 100/384 files...
Loaded 120/384 files...
Loaded 140/384 files...
Loaded 160/384 files...
Loaded 180/384 files...
Loaded 200/384 files...
Loaded 220/384 files...
Loaded 240/384 files...
Loaded 260/384 files...
Loaded 280/384 files...
Loaded 300/384 files...
Loaded 320/384 files...
Loaded 340/384 files...
Loaded 360/384 files...
Loaded 380/384 files...
Finished loading all 384 files. Total rows: 1,383,736


,count,mean,std,min,25%,50%,75%,max
Time;s,1383736.0,901.016346,520.990283,0.000000e+00,450.000000,9.005000e+02,1351.000000,2.540000e+03
ICE_Speed;rpm,1383736.0,1242.896959,1203.803406,0.000000e+00,0.000000,1.100218e+03,2173.352956,3.999294e+03
ICE_Torque;Nm,1383736.0,88.834564,112.297301,-9.330397e+01,0.000000,3.012032e+01,165.763555,5.226476e+02
ICE_Power;kW,1383736.0,20.833376,30.532738,-2.863298e+01,0.000000,4.133259e+00,32.419283,1.528551e+02
fuel_soll;mg,1383736.0,18.208601,21.247201,0.000000e+00,0.000000,9.010981e+00,33.797680,6.999873e+01
VTG;1,1383736.0,0.264985,0.271313,0.000000e+00,0.000000,2.300000e-01,0.447070,9.000000e-01
p_amb;bar,1383736.0,0.954294,0.084828,8.001663e-01,0.887227,9.542290e-01,1.023710,1.099220e+00
T_amb;K,1383736.0,281.953634,16.890825,2.501130e+02,267.788120,2.832650e+02,295.876000,3.099033e+02
NO_in_m;gps,1383736.0,0.038856,0.062007,0.000000e+00,0.000000,7.661099e-03,0.049763,3.832727e-01
NO2_in_m;gps,1383736.0,0.003134,0.005002,0.000000e+00,0.000000,6.179778e-04,0.004014,3.091645e-02


## 5. Advanced Data Analysis (Modular Skeleton)

Here you can add more advanced analytical functions, such as distribution plotting, correlation matrix generation, and identifying anomalies.

In [ ]:
def plot_column_distribution(filepath, column_name):
    """Plots the distribution of a single column from a given file."""
    df = load_and_preprocess_csv(filepath)
    if df is not None and column_name in df.columns:
        plt.figure(figsize=(10, 5))
        sns.histplot(df[column_name].dropna(), kde=True, bins=50)
        plt.title(f"Distribution of {column_name}")
        plt.xlabel(column_name)
        plt.ylabel("Frequency")
        plt.show()
    else:
        print(f"Column '{column_name}' not found or dataframe is empty.")
        print(f"Available columns: {list(df.columns) if df is not None else 'None'}")

def plot_correlation_matrix(filepath, top_n=15):
    """Plots a correlation matrix for the top N most variable columns."""
    df = load_and_preprocess_csv(filepath)
    if df is not None and not df.empty:
        # Select top N columns by variance to keep the plot legible
        variances = df.var().sort_values(ascending=False)
        top_cols = variances.index[:top_n]
        
        plt.figure(figsize=(12, 10))
        corr = df[top_cols].corr()
        sns.heatmap(corr, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
        plt.title("Correlation Matrix (Top Variance Columns)")
        plt.show()

In [ ]:
# Example: Plot distribution
if drivetrain_files:
    sample_df = load_and_preprocess_csv(drivetrain_files[0])
    col_to_plot = sample_df.columns[0] # Just picking the first column as an example
    print(f"Plotting {col_to_plot}...")
    plot_column_distribution(drivetrain_files[0], col_to_plot)

In [ ]:
# Example: Plot correlation matrix
if drivetrain_files:
    plot_correlation_matrix(drivetrain_files[0])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 5. Analyse starting temperatures and global T_amb;K column in Data_ICE files
ice_files = get_csv_files(DATA_ICE_DIR)
print(f"Found {len(ice_files)} files in {DATA_ICE_DIR}")

starting_temps = []
global_temps = []

for i, f in enumerate(ice_files):
    df = load_and_preprocess_csv(f)
    if df is not None and 'T_amb;K' in df.columns and not df.empty:
        # Starting temperature is the first reading
        starting_temps.append(df['T_amb;K'].iloc[0])
        # Global column data
        global_temps.append(df['T_amb;K'])

starting_temps_series = pd.Series(starting_temps, name='Starting T_amb')

print("\n--- Starting Temperatures (T_amb;K) ---")
print(f"Count: {starting_temps_series.count()}")
print(f"Min:  {starting_temps_series.min():.4f} K")
print(f"Max:  {starting_temps_series.max():.4f} K")
print(f"Mean: {starting_temps_series.mean():.4f} K")
print(f"Std:  {starting_temps_series.std():.4f} K")

if global_temps:
    all_tamb = pd.concat(global_temps, ignore_index=True)
    print("\n--- Global T_amb;K Column (All Time Steps) ---")
    print(f"Count: {all_tamb.count()}")
    print(f"Min:  {all_tamb.min():.4f} K")
    print(f"Max:  {all_tamb.max():.4f} K")
    print(f"Mean: {all_tamb.mean():.4f} K")
    print(f"Std:  {all_tamb.std():.4f} K")
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    sns.histplot(starting_temps_series, kde=True, bins=30, ax=axes[0], color='blue')
    axes[0].set_title('Distribution of Starting Temperatures')
    axes[0].set_xlabel('T_amb [K]')
    axes[0].set_ylabel('Frequency')
    
    sns.histplot(all_tamb, kde=True, bins=50, ax=axes[1], color='orange')
    axes[1].set_title('Global Distribution of T_amb;K (All Rows)')
    axes[1].set_xlabel('T_amb [K]')
    axes[1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
else:
    print("No T_amb;K data found.")
